# 3. Prediction on New Slides

`stp.predict(data, train_data=...)` runs inference on slide-image-only data
— no ground-truth expression, and no data config required for the WSI-path
forms. A checkpoint must be resolvable via `ckpt_path`, or via `train_data`
(or a prior `train()`/`from_run()` call on this instance).


> **Prerequisites**
> - STP-Bench installed (`bash scripts/create_env.sh`, see the
>   [README](../README.md#installation)).
> - Benchmark data downloaded for the dataset(s) used below (see
>   [README — Benchmark Data](../README.md#benchmark-data)), or your own
>   dataset added following
>   [docs/guide.md — Adding a New Dataset](../docs/guide.md#adding-a-new-dataset).
> - Run this notebook from the repo root, or pass `repo_root=` explicitly to
>   `STPred(...)`.


## The four kinds of prediction target

| Kind | Example | Needs `output_dir` |
|---|---|---|
| Named config | `"cptac/xenium"` | No |
| Single WSI file | `"/path/to/slide.svs"` | Yes |
| Directory of WSI files | `"/path/to/slides_dir/"` | Yes |
| Already-preprocessed asset dir (`patches/*.h5`) | `"/path/to/existing_assets"` | Yes |

The first is just `predict()` against a normal data config, same as
`evaluate_external()` without ground truth. The other three are covered
below — STP-Bench runs patch extraction and feature embedding automatically,
no data config to write.


In [ ]:
from stpbench import STPred

stp = STPred(models=["StNet"], repo_root=".")
stp.preprocess(data="ncche/xenium")
stp.train(data="ncche/xenium")


### A single slide file

In [ ]:
stp.predict(
    data="/path/to/slide.svs",
    output_dir="/path/to/output",
    train_data="ncche/xenium",   # or ckpt_path="/path/to/checkpoint.ckpt"
)


### A directory of slides

One prediction per slide, sharing `output_dir`. `wsi_dir` overrides where
the *original* slide files are looked up from later (e.g. by `visualize()`
in notebook 4) if it differs from the directory passed as `data`.


In [ ]:
stp.predict(data="/path/to/slides_dir/", output_dir="/path/to/output", train_data="ncche/xenium")


### An already-preprocessed asset directory

`output_dir` is where predictions (and `ids.csv`/embeddings, if a chosen
model needs embeddings not already present) are written. The asset
directory itself is only ever read — safe to point at something shared or
read-only.


In [ ]:
stp.predict(data="/path/to/existing_assets", output_dir="/path/to/predictions", train_data="ncche/xenium")


## Restricting output to specific genes

Names not found in the training gene panel are dropped with a warning
rather than failing the whole run; `ValueError` is raised only if *none* of
the requested names are found.


In [ ]:
stp.predict(
    data="/path/to/slide.svs", output_dir="/path/to/output", train_data="ncche/xenium",
    gene_list=["GENE1", "GENE2"],
)


## Per-call model override

Doesn't mutate `stp.models` — only affects this call.


In [ ]:
stp.predict(data="/path/to/slide.svs", output_dir="/path/to/output", train_data="ncche/xenium", models=["TRIPLEX"])


## Forcing re-extraction

If `output_dir` already has patches/embeddings from a previous call,
`overwrite=True` re-extracts everything from scratch rather than reusing
them.


In [ ]:
stp.predict(data="/path/to/slide.svs", output_dir="/path/to/output", train_data="ncche/xenium", overwrite=True)


## Predicting at your own patch coordinates

`coordinates` takes patch **centers** — an `.h5ad` (read from
`obsm['spatial']`) or `.csv` (`x`/`y` columns) — and crops exactly those
locations instead of running tissue segmentation + automatic tiling. Only
valid when `data` is a single WSI file.

If `output_dir` already has patches extracted by an earlier call against the
same slide, pass `overwrite=True` too — otherwise the old patches are reused
unchanged rather than re-cropped at the new coordinates.


In [ ]:
stp.predict(
    data="/path/to/slide.svs", output_dir="/path/to/output", train_data="ncche/xenium",
    coordinates="/path/to/spots.h5ad",   # or "/path/to/spots.csv"
)


## Batch size

`batch_size` (default 32) overrides how many patches are batched together
per model forward pass — the data config's own (usually `1`) is only used if
`batch_size=None` is passed explicitly. Raise or lower depending on GPU
memory and slide size.


## Coverage note

This WSI-path form works for models using the default `feature_type`
mechanism (StNet, TRIPLEX, DeepSpot, HisToGene, DeepSpotM, ...). Models with
`extra_preprocess` (EGN, EGGN, Sepal, OmiCLIP, M2ORT, M2OST) and SGN/Stem
still require the named-config path.
